# GRPO Training for CausalRepair

End-to-end pipeline that trains an LLM agent on the **CausalRepair** environment using **GRPO** (Group Relative Policy Optimization) — *not* PPO.

The notebook follows five phases:

| Phase | Section | Purpose |
|------:|:--------|:--------|
| **A** | Baseline | Zero-shot evaluation of the base model on real episodes |
| **B** | Data Collection | Stochastic rollouts to build a GRPO prompt dataset and inspect reward signal |
| **C** | GRPO Training | TRL `GRPOTrainer` + Unsloth 4-bit QLoRA, with an **episode-rollout reward** backed by your existing `compute_reward()` shaping |
| **D** | Evaluation | Run the trained adapter on fresh episodes and compare against baseline |
| **E** | Visualization | Training curves + before-vs-after reward and step charts |

**Design notes**

- Reward is computed by **rolling out the model's full multi-action completion** in a fresh `CausalrepairEnvironment`, summing per-step shaped rewards from `compute_reward()` — so `+1.0` (successful commit) is reachable and all your shaping (`-0.05` diagnose, `+0.05*newly_known` propagate, `-0.1` post-healthy step, `-1.0` bad commit, `-0.5` timeout, efficiency/budget bonuses) is preserved.
- All generation is **stochastic** (`do_sample=True`, `temperature=0.7`, `top_p=0.9`) — never greedy.
- GRPO has **no value head**. The "training curves" figure plots policy loss + KL divergence + mean reward in place of policy / value / reward.


## Setup

### Colab usage
On a fresh Colab runtime (Runtime > Change runtime type > **T4 GPU**):

```bash
!git clone https://huggingface.co/spaces/<your-user>/CausalRepair /content/CausalRepair_repo
%cd /content/CausalRepair_repo/CausalRepair
!pip -q install -r requirements.txt
```

The cells below take care of everything else (Unsloth, TRL, datasets, plotting, etc.).


In [ ]:
# Install Unsloth (4-bit QLoRA + fast LoRA), TRL (GRPOTrainer), datasets, plotting libs.
# On Colab T4 the unsloth[colab-new] extra pulls the right CUDA wheels for you.
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install "trl>=0.12" "transformers>=4.46" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.44"
!pip -q install datasets matplotlib seaborn tqdm pydantic


In [ ]:
# Clone the Space (flat layout: inference.py / server/ / requirements.txt
# live at the repo root, no nested CausalRepair/ folder).

!rm -rf /content/CausalRepair_repo
!git clone https://huggingface.co/spaces/Himanshu24Sharma/CausalRepair /content/CausalRepair_repo
%cd /content/CausalRepair_repo
!pip -q install -r requirements.txt


In [ ]:
import os, sys, json, re, copy, random, warnings, statistics
from pathlib import Path

# Make the CausalRepair package importable exactly like inference.py does
# (it relies on `from models import ...` and `from server... import ...`).
PACKAGE_ROOT = Path.cwd()
if PACKAGE_ROOT.name != "CausalRepair":
    candidate = PACKAGE_ROOT / "CausalRepair"
    if candidate.exists():
        PACKAGE_ROOT = candidate
        os.chdir(PACKAGE_ROOT)
sys.path.insert(0, str(PACKAGE_ROOT))
print("Package root:", PACKAGE_ROOT)

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# -------------------------------------------------------------------------
# Top-level configuration — tweak here.
# -------------------------------------------------------------------------
BASE_MODEL = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"

BASELINE_EPISODES     = 20    # Phase A
ROLLOUT_EPISODES      = 150   # Phase B
MAX_STEPS_PER_EPISODE = 12
GRPO_UPDATES          = 200   # Phase C: number of optimizer updates
NUM_GENERATIONS       = 4     # GRPO group size (completions per prompt)
EVAL_EPISODES         = 10    # Phase D

SAMPLING = dict(do_sample=True, temperature=0.7, top_p=0.9, max_new_tokens=256)

ADAPTER_DIR = "output/grpo_adapter"

random.seed(0)


In [ ]:
# Reuse the EXACT environment, adapter, reward function and system prompt that
# the existing repo / inference.py uses. No reward logic is reimplemented.
from models import CausalrepairAction, CausalrepairObservation
from server.CausalRepair_environment import CausalrepairEnvironment
from server.code_repair_adapter import CodeRepairAdapter
from inference import compute_reward, _world_healthy, build_prompt, SYSTEM_PROMPT

print("compute_reward      :", compute_reward.__module__)
print("_world_healthy      :", _world_healthy.__module__)
print("SYSTEM_PROMPT chars :", len(SYSTEM_PROMPT))


## Model loading (Unsloth 4-bit + LoRA)

We load `unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit` in 4-bit, attach a small LoRA (r=8) so the same model object is used for **baseline**, **data collection**, and **GRPO training**. After training we just flip it into inference mode for evaluation; no manual re-loading is needed.


In [ ]:
import torch
from unsloth import FastLanguageModel

assert torch.cuda.is_available(), "GRPO training expects a CUDA GPU (Colab T4 is fine)."

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

# Attach LoRA — small rank, all attention + MLP projections.
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=0,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Model loaded:", BASE_MODEL)
print("Trainable parameters:")
model.print_trainable_parameters()


## Phase A — Baseline Evaluation

Mirrors the loop in `inference.py:main()` exactly (including the *force-commit-when-healthy* branch from lines 242-278), but uses our local Unsloth model instead of an OpenAI-compatible API. Generation is **stochastic** (no greedy) so every phase shares the same exploration regime.

For each of `BASELINE_EPISODES` episodes we record the total reward, step count, and a `success` flag (commit was the terminating action AND `info.constraints_ok` is true). These lists are kept around for the comparison plots in Phase E.


In [ ]:
from tqdm.auto import tqdm


def make_env() -> CausalrepairEnvironment:
    """Factory that creates a fresh CausalRepair environment.

    The env's reset() seeds a deterministic broken `add` function, so every
    episode starts from the same state — exactly what Phase A / B / D / the
    GRPO reward function all share.
    """
    return CausalrepairEnvironment(adapter=CodeRepairAdapter())


_JSON_OBJ_RE = re.compile(r"\{.*?\}", re.DOTALL)


def _coerce_action(text: str) -> CausalrepairAction:
    """Parse a single CausalrepairAction out of free-form model text.

    Tries strict JSON first, then falls back to the first {...} block. If
    nothing parses we return a `commit_repair` action (same fallback as
    inference.py) so the episode terminates rather than silently looping.
    """
    text = (text or "").strip()
    candidates = []
    if text:
        candidates.append(text)
        match = _JSON_OBJ_RE.search(text)
        if match:
            candidates.append(match.group(0))
    for cand in candidates:
        try:
            payload = json.loads(cand)
            if isinstance(payload, dict) and "action_type" in payload:
                return CausalrepairAction(**payload)
        except Exception:
            continue
    return CausalrepairAction(action_type="commit_repair")


def generate_action(model, tokenizer, prompt: str, sampling=None) -> tuple:
    """Run the local Unsloth model once, return (CausalrepairAction, raw_text).

    Always sampling-on (do_sample=True). Uses the chat template so the system
    prompt the LLM sees is identical to the one in inference.py.
    """
    sampling = sampling or SAMPLING
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)
    with torch.inference_mode():
        out = model.generate(
            input_ids=inputs,
            pad_token_id=tokenizer.pad_token_id,
            **sampling,
        )
    raw = tokenizer.decode(out[0, inputs.shape[-1]:], skip_special_tokens=True)
    return _coerce_action(raw), raw


def run_episode(model, tokenizer, max_steps: int, record_steps: bool = False):
    """Run one full episode and return (total_reward, num_steps, success, step_records).

    `step_records` is populated only when record_steps=True (used by the data
    collector in Phase B; baseline / eval pass False to keep memory small).
    """
    env = make_env()
    obs = env.reset().observation
    done = False
    step_count = 0
    total_reward = 0.0
    last_intervened_entity = None
    last_intervened_value = None
    last_action_type = None
    last_constraints_ok = False
    step_records: list = []

    while not done and step_count < max_steps:
        was_healthy_before = _world_healthy(obs)
        prompt = build_prompt(obs)
        action, raw_text = generate_action(model, tokenizer, prompt)
        if action.action_type == "intervene":
            last_intervened_entity = action.target
            last_intervened_value = action.value
        try:
            step_result = env.step(action)
            obs = step_result.observation
            done = step_result.done
            info = step_result.info
            reward = compute_reward(
                action=action,
                done=done,
                info=info,
                max_steps=env.max_steps,
                diagnose_budget=env.diagnose_budget,
                was_healthy_before=was_healthy_before,
            )
        except Exception:
            obs, reward, done, info = None, 0.0, True, {}

        total_reward += reward
        step_count += 1
        last_action_type = action.action_type
        last_constraints_ok = bool((info or {}).get("constraints_ok", False))
        if record_steps:
            step_records.append((prompt, raw_text, float(reward)))

        # Force-commit when the world becomes healthy mid-episode (matches
        # inference.py lines 242-278). This keeps the agent honest: it cannot
        # spin forever after a successful repair.
        if not done and _world_healthy(obs) and step_count < max_steps:
            commit_action = CausalrepairAction(
                action_type="commit_repair",
                target=last_intervened_entity,
                value=last_intervened_value,
                rationale="All tests pass; fix is minimal and correct.",
            )
            try:
                step_result = env.step(commit_action)
                obs = step_result.observation
                done = step_result.done
                info = step_result.info
                reward = compute_reward(
                    action=commit_action,
                    done=done,
                    info=info,
                    max_steps=env.max_steps,
                    diagnose_budget=env.diagnose_budget,
                    was_healthy_before=True,
                )
            except Exception:
                reward, done = 0.0, True
                info = {}
            total_reward += reward
            step_count += 1
            last_action_type = commit_action.action_type
            last_constraints_ok = bool((info or {}).get("constraints_ok", False))
            if record_steps:
                step_records.append((build_prompt(obs), str(commit_action), float(reward)))
            break

    success = bool(last_action_type == "commit_repair" and last_constraints_ok)
    return total_reward, step_count, success, step_records


def run_baseline(model, tokenizer, n_episodes: int, max_steps: int = MAX_STEPS_PER_EPISODE):
    rewards, steps, successes = [], [], []
    FastLanguageModel.for_inference(model)
    for _ in tqdm(range(n_episodes), desc="baseline"):
        r, s, ok, _ = run_episode(model, tokenizer, max_steps=max_steps)
        rewards.append(r)
        steps.append(s)
        successes.append(ok)
    avg_r = sum(rewards) / len(rewards)
    avg_s = sum(steps) / len(steps)
    succ_rate = sum(successes) / len(successes)
    print(f"[BASELINE] episodes={n_episodes}  avg_reward={avg_r:+.3f}  avg_steps={avg_s:.2f}  success_rate={succ_rate:.0%}")
    return rewards, steps, successes


In [ ]:
baseline_rewards, baseline_steps, baseline_success = run_baseline(
    model, tokenizer, n_episodes=BASELINE_EPISODES,
)


## Phase B — Data Collection

Run **stochastic** rollouts with the same Unsloth model + the same env + the same shaped reward. This serves two purposes:

1. **Diagnostics**: per-step `(prompt, response, reward)` tuples + per-episode totals + success rate. We log mean / min / max / non-zero proportion / success proportion so you can immediately see whether the reward signal is informative enough for GRPO.
2. **GRPO prompt distribution**: we capture the **initial observation text** (constant across resets in this deterministic env) and use it as the GRPO training prompt distribution. With `num_generations=4`, GRPO will draw 4 distinct multi-action trajectories per training step and rank them by total reward — that's the entire point of *group-relative* advantage.

We do not change the env / adapter to inflate signal — we only scale `ROLLOUT_EPISODES` and rely on stochastic sampling + your existing shaping.


In [ ]:
def collect_data(model, tokenizer, num_episodes: int, max_steps: int):
    """Stochastic rollouts -> (prompts, responses, rewards, episode_totals, episode_success).

    All four lists at index i represent step i of the flattened rollout (per-step granularity).
    `episode_totals` and `episode_success` are length num_episodes.
    """
    prompts: list = []
    responses: list = []
    rewards: list = []
    episode_totals: list = []
    episode_steps: list = []
    episode_success: list = []

    FastLanguageModel.for_inference(model)
    for _ in tqdm(range(num_episodes), desc="collect_data"):
        total, n_steps, ok, step_records = run_episode(
            model, tokenizer, max_steps=max_steps, record_steps=True,
        )
        for (p, r_text, r) in step_records:
            prompts.append(p)
            responses.append(r_text)
            rewards.append(r)
        episode_totals.append(total)
        episode_steps.append(n_steps)
        episode_success.append(ok)
    return prompts, responses, rewards, episode_totals, episode_steps, episode_success


def report_data(prompts, responses, rewards, ep_totals, ep_success):
    n_samples = len(prompts)
    n_eps = len(ep_totals)
    if n_samples == 0:
        print("[COLLECT] no samples collected")
        return
    mean_r = sum(rewards) / n_samples
    min_r = min(rewards)
    max_r = max(rewards)
    std_r = statistics.pstdev(rewards) if n_samples > 1 else 0.0
    nonzero_frac = sum(1 for r in rewards if r != 0.0) / n_samples
    succ_frac = sum(1 for ok in ep_success if ok) / n_eps if n_eps else 0.0
    mean_ep_total = sum(ep_totals) / n_eps if n_eps else 0.0

    print(f"[COLLECT] samples={n_samples}  episodes={n_eps}")
    print(f"[COLLECT] step reward: mean={mean_r:+.3f}  std={std_r:.3f}  min={min_r:+.3f}  max={max_r:+.3f}")
    print(f"[COLLECT] non-zero reward fraction = {nonzero_frac:.1%}")
    print(f"[COLLECT] mean episode return       = {mean_ep_total:+.3f}")
    print(f"[COLLECT] episodes ending in success = {succ_frac:.1%}")
    if n_samples < 500:
        warnings.warn(
            f"Collected only {n_samples} samples (<500). Consider increasing "
            f"ROLLOUT_EPISODES — sparse-reward GRPO needs diverse exploration.",
            stacklevel=1,
        )


prompts, responses, rewards, episode_totals, episode_steps_arr, episode_success_arr = collect_data(
    model, tokenizer, num_episodes=ROLLOUT_EPISODES, max_steps=MAX_STEPS_PER_EPISODE,
)
report_data(prompts, responses, rewards, episode_totals, episode_success_arr)


In [ ]:
# Build the GRPO prompt distribution. The initial observation is deterministic
# (always the same broken `add` function), so we use one canonical initial-state
# prompt repeated ROLLOUT_EPISODES times. Diversity then comes from
# num_generations distinct sampled trajectories per group, NOT from the
# prompt text itself.
_canon_env = make_env()
_canon_obs = _canon_env.reset().observation
INITIAL_PROMPT_TEXT = build_prompt(_canon_obs)
episode_initial_prompts = [INITIAL_PROMPT_TEXT for _ in range(max(ROLLOUT_EPISODES, 64))]

print("Initial-prompt sample (first 400 chars):")
print(INITIAL_PROMPT_TEXT[:400])
print("...")
print(f"Total GRPO training prompts: {len(episode_initial_prompts)}")


## Phase C — GRPO Training

We use TRL's **`GRPOTrainer`** (canonical GRPO API: a reward callback + `trainer.train()`; *not* a manual `step(queries, responses, rewards)` loop, which is the PPO API and doesn't exist on `GRPOTrainer`).

### Reward function: episode rollout

Each completion is parsed into a sequence of JSON action objects. We instantiate a fresh `CausalrepairEnvironment`, reset it, then step through the parsed actions. After every step we call the **same `compute_reward()`** that powers `inference.py` (with the same `was_healthy_before`, `max_steps`, `diagnose_budget` signals), and accumulate the per-step shaped rewards into the trajectory's total return. That total is the GRPO reward.

This makes the success bonus (`+1.0 + efficiency + budget`) reachable, and preserves all the existing intermediate shaping (`-0.05` diagnose, `+0.05*newly_known` propagate, `-0.1` post-healthy step penalty, `-1.0` bad commit, `-0.5` timeout).

### GRPO-specific system prompt

We override the single-action `SYSTEM_PROMPT` with a **multi-action** variant that asks the model to emit one JSON action per line until `commit_repair`. The original `SYSTEM_PROMPT` from `inference.py` stays untouched and continues to drive Phase A / B / D.


In [ ]:
GRPO_SYSTEM_PROMPT = (
    SYSTEM_PROMPT
    + "\n\nFor THIS task you must output a SEQUENCE of actions, one per line. "
    "Each line is a single JSON object with the same schema as above. "
    "Do not include any other text, no markdown, no commentary, no blank lines. "
    "End the sequence with exactly one commit_repair line."
)


def parse_actions(text: str, max_actions: int = MAX_STEPS_PER_EPISODE):
    """Tolerant parser: extract up to `max_actions` JSON action objects.

    Strategy:
      1. Try every line as JSON (handles the canonical one-per-line format).
      2. If that yielded nothing, scan with a non-greedy {...} regex over the
         whole completion (handles models that concatenate or wrap in prose).
    """
    actions: list = []
    if not text:
        return actions

    for line in text.splitlines():
        line = line.strip().strip(",")
        if not line:
            continue
        try:
            payload = json.loads(line)
        except Exception:
            continue
        if isinstance(payload, dict) and "action_type" in payload:
            try:
                actions.append(CausalrepairAction(**payload))
            except Exception:
                continue
        if len(actions) >= max_actions:
            break

    if not actions:
        for match in _JSON_OBJ_RE.finditer(text):
            try:
                payload = json.loads(match.group(0))
            except Exception:
                continue
            if isinstance(payload, dict) and "action_type" in payload:
                try:
                    actions.append(CausalrepairAction(**payload))
                except Exception:
                    continue
            if len(actions) >= max_actions:
                break
    return actions


def reward_episode_rollout(completions, prompts=None, **kwargs):
    """GRPO reward callback. Returns a list of floats, one per completion.

    Each completion is rolled out in a fresh env via the same shaped reward
    function (`compute_reward`) used in Phase A and Phase B. If parsing fails
    or no actions are produced, the trajectory gets the -0.5 timeout penalty.
    """
    rewards_out: list = []
    for completion in completions:
        # GRPOTrainer can hand us either a plain string or a chat-formatted
        # list-of-messages — handle both.
        if isinstance(completion, list):
            text = "".join(m.get("content", "") for m in completion if isinstance(m, dict))
        else:
            text = str(completion)

        actions = parse_actions(text)
        env = make_env()
        reset_result = env.reset()
        obs = reset_result.observation
        total = 0.0
        done = False

        if not actions:
            rewards_out.append(-0.5)
            continue

        for action in actions:
            if done:
                break
            was_healthy_before = _world_healthy(obs)
            try:
                step_result = env.step(action)
                obs = step_result.observation
                done = step_result.done
                info = step_result.info
                r = compute_reward(
                    action=action,
                    done=done,
                    info=info,
                    max_steps=env.max_steps,
                    diagnose_budget=env.diagnose_budget,
                    was_healthy_before=was_healthy_before,
                )
            except Exception:
                r, done = -0.5, True
                info = {}
            total += r

        # If the model never issued commit_repair we treat it as a non-commit
        # timeout — same convention as compute_reward's `done and not commit`
        # branch.
        if not done:
            total += -0.5
        rewards_out.append(float(total))
    return rewards_out


# Smoke-test the reward function on the baseline responses we already have.
_demo_completions = [
    "\n".join([
        '{"action_type": "diagnose", "target": "add"}',
        '{"action_type": "intervene", "target": "add", "value": "def add(x, y):\\n    return x + y\\n"}',
        '{"action_type": "propagate"}',
        '{"action_type": "commit_repair", "target": "add", "value": "def add(x, y):\\n    return x + y\\n", "rationale": "fixed sign bug"}',
    ]),
    '{"action_type": "commit_repair", "target": "add"}',
    "garbage non-JSON text",
]
print("Smoke-test rewards:", reward_episode_rollout(_demo_completions))


In [ ]:
from datasets import Dataset


def _to_chat_prompt(initial_obs_text: str):
    return [
        {"role": "system", "content": GRPO_SYSTEM_PROMPT},
        {"role": "user", "content": initial_obs_text},
    ]


train_ds = Dataset.from_list(
    [{"prompt": _to_chat_prompt(p)} for p in episode_initial_prompts]
)
print("GRPO train dataset:", train_ds)
print("First row prompt[0]['role']:", train_ds[0]["prompt"][0]["role"])


In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback


class MetricCollector(TrainerCallback):
    """Capture per-step metrics from GRPOTrainer's logs into in-memory lists.

    GRPOTrainer logs `loss`, `reward`, `kl`, `completions/mean_length`, ... at
    each optimizer step. We mirror them here so the visualization cell can
    plot training curves without needing TensorBoard / wandb.
    """

    def __init__(self):
        self.policy_losses: list = []
        self.mean_rewards: list = []
        self.kl_values: list = []
        self.completion_lengths: list = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        if "loss" in logs:
            self.policy_losses.append(float(logs["loss"]))
        if "reward" in logs:
            self.mean_rewards.append(float(logs["reward"]))
        if "kl" in logs:
            self.kl_values.append(float(logs["kl"]))
        if "completions/mean_length" in logs:
            self.completion_lengths.append(float(logs["completions/mean_length"]))


metrics = MetricCollector()

# GRPO requires per_device_train_batch_size to be divisible by num_generations.
PER_DEVICE_BATCH = max(NUM_GENERATIONS, 8)
assert PER_DEVICE_BATCH % NUM_GENERATIONS == 0

grpo_config = GRPOConfig(
    output_dir="output/grpo_checkpoints",
    learning_rate=5e-5,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=1,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=512,
    max_completion_length=512,
    max_steps=GRPO_UPDATES,
    temperature=0.7,
    top_p=0.9,
    beta=0.04,
    logging_steps=1,
    save_steps=GRPO_UPDATES,
    save_strategy="no",
    bf16=False,
    fp16=True,
    report_to="none",
    remove_unused_columns=False,
    seed=0,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_episode_rollout],
    args=grpo_config,
    train_dataset=train_ds,
    callbacks=[metrics],
)
print("GRPOTrainer ready. Starting training.")


In [ ]:
trainer.train()
print(f"Done. Captured {len(metrics.policy_losses)} loss points, "
      f"{len(metrics.mean_rewards)} reward points, "
      f"{len(metrics.kl_values)} KL points.")


In [ ]:
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA adapter saved to {ADAPTER_DIR}/")


## Phase D — Evaluation

We evaluate the *same* model object (now LoRA-trained) using the **same loop** as the baseline (`run_episode` with the single-action `SYSTEM_PROMPT`, force-commit branch on health). This keeps every variable except the policy weights identical between baseline and trained numbers.

If you want to load from disk in a fresh runtime instead of using the in-memory model, the alternative path is documented in the second cell.


In [ ]:
def evaluate_model(model, tokenizer, n_episodes: int, max_steps: int = MAX_STEPS_PER_EPISODE):
    """Same loop as run_baseline but with the post-GRPO model. Returns
    (rewards, steps, success) lists of length n_episodes.
    """
    rewards, steps, successes = [], [], []
    FastLanguageModel.for_inference(model)
    for _ in tqdm(range(n_episodes), desc="evaluate"):
        r, s, ok, _ = run_episode(model, tokenizer, max_steps=max_steps)
        rewards.append(r)
        steps.append(s)
        successes.append(ok)
    return rewards, steps, successes


trained_rewards, trained_steps, trained_success = evaluate_model(
    model, tokenizer, n_episodes=EVAL_EPISODES,
)


def _summary(label, rewards, steps, successes):
    print(
        f"{label:>10}  episodes={len(rewards):>3}  "
        f"avg_reward={sum(rewards)/len(rewards):+.3f}  "
        f"avg_steps={sum(steps)/len(steps):.2f}  "
        f"success_rate={sum(successes)/len(successes):.0%}"
    )


print("=" * 64)
_summary("baseline", baseline_rewards, baseline_steps, baseline_success)
_summary("trained",  trained_rewards,  trained_steps,  trained_success)
print("=" * 64)


In [ ]:
# OPTIONAL: reload the trained LoRA adapter from disk in a fresh runtime.
# Uncomment when you want to evaluate without re-training.
#
# from unsloth import FastLanguageModel
# from peft import PeftModel
# base, tok = FastLanguageModel.from_pretrained(
#     model_name=BASE_MODEL, max_seq_length=2048, load_in_4bit=True,
# )
# trained = PeftModel.from_pretrained(base, ADAPTER_DIR)
# FastLanguageModel.for_inference(trained)
# # Optional: merge LoRA into base weights for faster inference (one-way).
# # trained = trained.merge_and_unload()
# trained_rewards, trained_steps, trained_success = evaluate_model(
#     trained, tok, n_episodes=EVAL_EPISODES,
# )


## Phase E — Visualizations

Three figures saved under `output/`:

- `grpo_training_curves.png` — policy loss, KL divergence, and mean reward across optimizer steps. (KL replaces "value loss" because GRPO has no critic — it uses group-relative advantages directly.)
- `before_after_rewards.png` — per-episode reward + per-episode steps, baseline vs trained.
- `reward_histograms.png` — overlaid reward distributions before vs after.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
OUTPUT_DIR.mkdir(exist_ok=True)


def _xrange(series):
    return list(range(1, len(series) + 1))


fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
if metrics.policy_losses:
    ax.plot(_xrange(metrics.policy_losses), metrics.policy_losses, color="C0")
ax.set_title("Policy loss")
ax.set_xlabel("update step")
ax.set_ylabel("loss")

ax = axes[1]
if metrics.kl_values:
    ax.plot(_xrange(metrics.kl_values), metrics.kl_values, color="C1")
ax.set_title("KL divergence\n(GRPO has no value head; KL shown instead)")
ax.set_xlabel("update step")
ax.set_ylabel("KL")

ax = axes[2]
if metrics.mean_rewards:
    ax.plot(_xrange(metrics.mean_rewards), metrics.mean_rewards, color="C2")
ax.set_title("Mean reward")
ax.set_xlabel("update step")
ax.set_ylabel("reward")

fig.suptitle("GRPO training curves", y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "grpo_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.plot(_xrange(baseline_rewards), baseline_rewards, marker="o", label="baseline")
ax.plot(_xrange(trained_rewards),  trained_rewards,  marker="s", label="trained")
ax.set_title("Per-episode total reward")
ax.set_xlabel("episode index")
ax.set_ylabel("total reward")
ax.legend()

ax = axes[1]
ax.plot(_xrange(baseline_steps), baseline_steps, marker="o", label="baseline")
ax.plot(_xrange(trained_steps),  trained_steps,  marker="s", label="trained")
ax.set_title("Per-episode steps")
ax.set_xlabel("episode index")
ax.set_ylabel("steps")
ax.legend()

fig.suptitle("Before vs after GRPO", y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "before_after_rewards.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
bins = 20
ax.hist(baseline_rewards, bins=bins, alpha=0.6, label="baseline", color="C0")
ax.hist(trained_rewards,  bins=bins, alpha=0.6, label="trained",  color="C2")
ax.axvline(x=sum(baseline_rewards) / max(1, len(baseline_rewards)), linestyle="--", color="C0", linewidth=1)
ax.axvline(x=sum(trained_rewards)  / max(1, len(trained_rewards)),  linestyle="--", color="C2", linewidth=1)
ax.set_title("Episode reward distribution: baseline vs trained")
ax.set_xlabel("total episode reward")
ax.set_ylabel("episode count")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "reward_histograms.png", dpi=150, bbox_inches="tight")
plt.show()


## Optional — CLI shell-out for baseline metrics

`inference.py` has been extended with a `--json` mode that prints exactly one JSON line per episode (`{"reward": float, "steps": int, "success": bool}`) and suppresses every other stdout line. You can call it from the notebook to get baseline numbers via the OpenAI-compatible HF router instead of the local Unsloth model.

This requires `HF_TOKEN` (or `API_KEY`) to be set in the environment.


In [ ]:
import subprocess

# Skip if no HF token is configured.
if os.getenv("HF_TOKEN") or os.getenv("API_KEY"):
    proc = subprocess.run(
        ["python", "inference.py", "--episodes", "5", "--json", "--max-steps", "12"],
        cwd=str(PACKAGE_ROOT),
        capture_output=True, text=True, check=False,
    )
    print("stderr:", proc.stderr[-500:] if proc.stderr else "(empty)")
    cli_rewards, cli_steps, cli_success = [], [], []
    for line in proc.stdout.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue
        cli_rewards.append(float(row["reward"]))
        cli_steps.append(int(row["steps"]))
        cli_success.append(bool(row["success"]))
    if cli_rewards:
        print(f"[CLI baseline] avg_reward={sum(cli_rewards)/len(cli_rewards):+.3f}  "
              f"avg_steps={sum(cli_steps)/len(cli_steps):.2f}  "
              f"success_rate={sum(cli_success)/len(cli_success):.0%}  "
              f"(episodes={len(cli_rewards)})")
    else:
        print("CLI did not emit any JSON lines. Inspect proc.stdout for diagnostics:")
        print(proc.stdout[-500:])
else:
    print("HF_TOKEN / API_KEY not set — skipping CLI shell-out demo.")
